# Baseline Holt-Winters (GCP Spark on YARN)
- Input: dense 30-minute demand (from feature engineering)
- Train/val/test: use split column from HDFS
- Model: Holt-Winters per zone (seasonal=48 bins/day)
- Metrics: RMSE, MAE, MAPE, sMAPE, R2
- Output: zone-level predictions and metrics to HDFS

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from statsmodels.tsa.holtwinters import ExponentialSmoothing

BASE_HDFS = "/user/tiennd"
DENSE_PATH = f"{BASE_HDFS}/feature_engineering/demand_prediction_dense_30m"
OUT_PRED = f"{BASE_HDFS}/results/baseline_holt_winters_gcp/predictions"
OUT_METRICS = f"{BASE_HDFS}/results/baseline_holt_winters_gcp/metrics"

BIN_COL = "pickup_bin_30m"
ZONE_COL = "PULocationID"
VALUE_COL = "pickup_demand"
SEASONAL_PERIODS = 48

spark = (
    SparkSession.builder
    .appName("BaselineHoltWinters_GCP")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", f"hdfs://{BASE_HDFS}/spark-logs")
    .config("spark.executor.instances", "3")
    .config("spark.executor.cores", "3")
    .config("spark.executor.memory", "6g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.memoryOverhead", "1g")
    .config("spark.sql.shuffle.partitions", "96")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

dense = (
    spark.read.parquet(DENSE_PATH)
    .select("split", ZONE_COL, BIN_COL, VALUE_COL)
)
print("Dense rows:", dense.count())

schema = StructType([
    StructField("split", StringType(), False),
    StructField(ZONE_COL, IntegerType(), False),
    StructField(BIN_COL, TimestampType(), False),
    StructField("y_true", DoubleType(), True),
    StructField("y_pred", DoubleType(), True),
])

def forecast_zone(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values(BIN_COL)
    zone_id = int(pdf[ZONE_COL].iloc[0])
    train = pdf[pdf["split"] == "train"]
    val = pdf[pdf["split"] == "val"]
    test = pdf[pdf["split"] == "test"]

    y_train = train[VALUE_COL].astype(float).values
    fallback_value = float(y_train[-1]) if len(y_train) else 0.0

    def safe_forecast(n_steps: int) -> np.ndarray:
        if n_steps <= 0:
            return np.array([], dtype=float)
        if len(y_train) < SEASONAL_PERIODS * 2:
            return np.full(n_steps, fallback_value, dtype=float)
        try:
            model = ExponentialSmoothing(
                y_train,
                trend="add",
                seasonal="add",
                seasonal_periods=SEASONAL_PERIODS,
            ).fit(optimized=True)
            return model.forecast(n_steps).astype(float)
        except Exception:
            return np.full(n_steps, fallback_value, dtype=float)

    val_pred = safe_forecast(len(val))
    test_pred = safe_forecast(len(test))

    out_frames = []
    if len(val) > 0:
        tmp = val[["split", ZONE_COL, BIN_COL, VALUE_COL]].copy()
        tmp["y_pred"] = val_pred
        tmp = tmp.rename(columns={VALUE_COL: "y_true"})
        out_frames.append(tmp)
    if len(test) > 0:
        tmp = test[["split", ZONE_COL, BIN_COL, VALUE_COL]].copy()
        tmp["y_pred"] = test_pred
        tmp = tmp.rename(columns={VALUE_COL: "y_true"})
        out_frames.append(tmp)

    if len(out_frames) == 0:
        return pd.DataFrame(columns=["split", ZONE_COL, BIN_COL, "y_true", "y_pred"])

    result = pd.concat(out_frames, ignore_index=True)
    result[ZONE_COL] = zone_id
    return result[["split", ZONE_COL, BIN_COL, "y_true", "y_pred"]]

predictions = (
    dense.groupBy(ZONE_COL)
    .applyInPandas(forecast_zone, schema=schema)
    .cache()
)
print("Pred rows:", predictions.count())

In [ ]:
metrics_base = (
    predictions
    .withColumn("err", F.col("y_true") - F.col("y_pred"))
    .withColumn("abs_err", F.abs(F.col("err")))
    .withColumn("sq_err", F.col("err") ** 2)
    .withColumn("abs_pct", F.when(F.col("y_true") != 0, F.abs(F.col("err") / F.col("y_true"))).otherwise(F.lit(None)))
    .withColumn("smape", F.when((F.abs(F.col("y_true")) + F.abs(F.col("y_pred"))) != 0,
                          2 * F.abs(F.col("err")) / (F.abs(F.col("y_true")) + F.abs(F.col("y_pred")))
                         ).otherwise(F.lit(None)))
)

agg = (
    metrics_base
    .groupBy("split")
    .agg(
        F.sqrt(F.avg("sq_err")).alias("rmse"),
        F.avg("abs_err").alias("mae"),
        F.avg("abs_pct").alias("mape"),
        F.avg("smape").alias("smape"),
        F.avg("y_true").alias("y_mean"),
    )
)

sse = metrics_base.groupBy("split").agg(F.sum("sq_err").alias("sse"))
sst = (
    metrics_base
    .join(agg.select("split", "y_mean"), on="split", how="left")
    .withColumn("sst", (F.col("y_true") - F.col("y_mean")) ** 2)
    .groupBy("split")
    .agg(F.sum("sst").alias("sst"))
)

metrics = (
    agg.join(sse, on="split", how="left")
    .join(sst, on="split", how="left")
    .withColumn("r2", F.when(F.col("sst") != 0, 1 - (F.col("sse") / F.col("sst"))).otherwise(F.lit(None)))
)
metrics.show(truncate=False)

predictions.write.mode("overwrite").parquet(OUT_PRED)
metrics.write.mode("overwrite").parquet(OUT_METRICS)
print("Saved to:", OUT_PRED)
print("Saved to:", OUT_METRICS)

spark.catalog.clearCache()
spark.stop()